# Evaluations — Russian Claims

This notebook explores `evaluations_russian_claims.csv`, the combined pairwise-comparison dataset from the counter-narrative (CN) pilot experiment.

## Column reference

| Column | Description |
|---|---|
| `submission_date` | Timestamp when the evaluator submitted the response |
| `response_id` | Unique sequential ID for each row |
| `evaluator_id` | Which human evaluator made the judgment |
| `baseclaim_id` | Which Russian claim was being counter-narrated |
| `left_narrative_id` | ID of the **left** counter-narrative |
| `left_rhetorical_id` | Rhetorical-technique ID of the **left** counter-narrative |
| `left_style_id` | Writing-style ID of the **left** counter-narrative |
| `right_narrative_id` | Same three attributes for the **right** counter-narrative |
| `right_rhetorical_id` |  |
| `right_style_id` |  |
| `kpi_id` | Which evaluation criterion is being judged (1, 2, or 3) |
| `left_chosen` | 1 = evaluator preferred the left CN; 0 = preferred right CN |
| `response_time` | Time in seconds to make the pairwise choice |

**Row grouping:** every comparison between a left/right pair appears as **3 consecutive rows** (one per KPI) with identical left/right attributes.

In [1]:
import pandas as pd

df = pd.read_csv('evaluations_russian_claims.csv')
print(f'Rows: {len(df):,}   Columns: {len(df.columns)}')
df.head(9)

Rows: 7,650   Columns: 13


,submission_date,response_id,evaluator_id,baseclaim_id,left_narrative_id,left_rhetorical_id,left_style_id,right_narrative_id,right_rhetorical_id,right_style_id,kpi_id,left_chosen,response_time
0,23/09/2024 07:47,1,1,1,111,12,1,129,13,9,1,1,59
1,23/09/2024 07:47,2,1,1,111,12,1,129,13,9,2,1,7
2,23/09/2024 07:47,3,1,1,111,12,1,129,13,9,3,1,24
3,23/09/2024 07:48,4,1,1,72,8,2,117,12,7,1,1,38
4,23/09/2024 07:48,5,1,1,72,8,2,117,12,7,2,1,8
5,23/09/2024 07:48,6,1,1,72,8,2,117,12,7,3,1,10
6,23/09/2024 07:49,7,1,1,36,4,6,94,10,4,1,1,42
7,23/09/2024 07:49,8,1,1,36,4,6,94,10,4,2,0,6
8,23/09/2024 07:49,9,1,1,36,4,6,94,10,4,3,0,5


## Experimental Setup

Pairwise comparisons were allocated via a non-overlapping random partition: each evaluator received a unique, disjoint set of 170 CN pairs per base claim, drawn from the 8,385 possible pairs over 130 CNs (~2% coverage per evaluator).


In [2]:
# Each triplet of rows (kpi_id 1-2-3, same left/right pair) is one comparison
comparisons = df[df['kpi_id'] == 1]
print(f'Total comparisons: {len(comparisons):,}')

summary = (
    comparisons
    .groupby(['evaluator_id', 'baseclaim_id'])
    .size()
    .reset_index(name='n_comparisons')
)
print()
print(summary.to_string(index=False))

Total comparisons: 2,550

 evaluator_id  baseclaim_id  n_comparisons
            1             1            170
            1             2            170
            1             3            170
            2             1            170
            2             2            170
            2             3            170
            3             1            170
            3             2            170
            3             3            170
            4             1            170
            4             2            170
            4             3            170
            5             1            170
            5             2            170
            5             3            170


---
## Internal Consistency

For each *(evaluator × base claim × KPI)* condition we derive each CN's **Copeland score** (wins − losses) from the evaluator's 170 observed choices. The **rank-biserial correlation** between this implied ranking and the individual pairwise choices measures how strongly an evaluator's own preferences are self-consistent (Cureton 1956).

- **r = 0**: no association between ranking and choices
- **r = 1**: every pairwise choice agrees with the implied ranking (perfect consistency)
- **|r| ≥ 0.50**: large effect size (Cohen 1988)

The function below computes Copeland scores, checks what fraction of observed pairs agree with the ranking, and converts that fraction to r via the standard rescaling: r = (raw − 0.5) × 2.


In [3]:
def rank_biserial(group):
    """
    Rank-biserial correlation between the Copeland-implied ranking and
    pairwise choices (Cureton 1956), adapted for sparse tournaments.

    Steps:
      1. Assign each CN a Copeland score = wins - losses (observed pairs only).
      2. For every observed pair, check whether the higher-scored CN was chosen.
      3. raw in [0.5, 1.0]; rescaled r = (raw - 0.5) * 2 in [0, 1].
    """
    wins, losses, outcomes = {}, {}, []
    for _, row in group.iterrows():
        a, b = int(row['left_narrative_id']), int(row['right_narrative_id'])
        w, l = (a, b) if int(row['left_chosen']) == 1 else (b, a)
        wins[w]   = wins.get(w, 0) + 1
        losses[l] = losses.get(l, 0) + 1
        outcomes.append((w, l))

    if not outcomes:
        return float('nan')

    items = set(wins) | set(losses)
    score = {i: wins.get(i, 0) - losses.get(i, 0) for i in items}

    n_correct = sum(
        1.0 if score[w] > score[l] else
        0.5 if score[w] == score[l] else
        0.0
        for w, l in outcomes
    )
    raw = n_correct / len(outcomes)   # in [0.5, 1.0]
    return round((raw - 0.5) * 2, 4)  # rescaled to [0, 1]


In [4]:
# Compute rank-biserial r for every (evaluator, base claim, KPI) condition
results = (
    df.groupby(['evaluator_id', 'baseclaim_id', 'kpi_id'])
    .apply(
        lambda g: pd.Series({
            'n_comparisons': len(g),
            'n_items':       len(set(g['left_narrative_id']) | set(g['right_narrative_id'])),
            'rank_biserial': rank_biserial(g),
        }),
        include_groups=False
    )
    .reset_index()
)

results

,evaluator_id,baseclaim_id,kpi_id,n_comparisons,n_items,rank_biserial
0,1,1,1,170.0,122.0,0.8000
1,1,1,2,170.0,122.0,0.8000
2,1,1,3,170.0,122.0,0.7118
3,1,2,1,170.0,121.0,0.7824
4,1,2,2,170.0,121.0,0.7882
5,1,2,3,170.0,121.0,0.7529
6,1,3,1,170.0,123.0,0.8059
7,1,3,2,170.0,123.0,0.7529
8,1,3,3,170.0,123.0,0.7706
9,2,1,1,170.0,124.0,0.7824


In [5]:
# Mean rank-biserial r per evaluator (averaged over 3 base claims and 3 KPIs)
print("=== Rank-Biserial r — mean per evaluator ===")
print("(0 = no association, 1 = perfect consistency)\n")
summary = results.groupby('evaluator_id')['rank_biserial'].agg(['mean', 'std', 'min', 'max']).round(4)
print(summary)

print("\n=== Rank-biserial r — breakdown by evaluator × KPI ===")
print(results.pivot_table(values='rank_biserial', index='evaluator_id', columns='kpi_id',
                           aggfunc='mean').round(4))


=== Rank-Biserial r — mean per evaluator ===
(0 = no association, 1 = perfect consistency)

                mean     std     min     max
evaluator_id                                
1             0.7739  0.0304  0.7118  0.8059
2             0.7595  0.0608  0.6588  0.8412
3             0.7843  0.0553  0.7000  0.8529
4             0.7830  0.0401  0.6882  0.8235
5             0.6516  0.0581  0.5765  0.7588

=== Rank-biserial r — breakdown by evaluator × KPI ===
kpi_id             1       2       3
evaluator_id                        
1             0.7961  0.7804  0.7451
2             0.8059  0.7118  0.7608
3             0.8216  0.8019  0.7294
4             0.8019  0.8000  0.7470
5             0.6451  0.6451  0.6647


---
## Inter-Rater Agreement — Bradley-Terry Model

Since no pairs are shared across evaluators, inter-rater agreement is measured indirectly via **ranking concordance**. For each *(base claim × KPI × evaluator)* condition we fit a **Bradley-Terry model** to the 170 pairwise choices, yielding a continuous latent quality score β per CN.

The model is fitted as logistic regression: for each comparison, build a feature vector with +1 at the left CN index and −1 at the right CN index, and predict `left_chosen`. The learned coefficients are the β scores. BT scores are continuous and adjust for opponent strength, yielding more reliable per-CN quality rankings from sparse data than raw win counts.

L2 regularisation (C = 1.0) prevents divergence for CNs that only ever won or only ever lost.


In [6]:
import numpy as np
from scipy.sparse import lil_matrix
from sklearn.linear_model import LogisticRegression


def fit_bradley_terry(group):
    """
    Fit a Bradley-Terry model via logistic regression.

    For each comparison (A vs B, A wins):
        feature vector = e_A - e_B  (+1 at A index, -1 at B index)
        label = 1

    For (A vs B, B wins):
        feature vector = e_A - e_B
        label = 0   (equivalently: B's coefficient gets pulled up)

    Returns a dict {cn_id: beta_score}.
    L2 regularisation (C=1.0) handles CNs that never lost / never won.
    """
    rows = list(group.itertuples(index=False))
    if not rows:
        return {}

    # Build ordered CN index
    all_cns = sorted(set(int(r.left_narrative_id) for r in rows) |
                     set(int(r.right_narrative_id) for r in rows))
    cn_idx = {cn: i for i, cn in enumerate(all_cns)}
    n_cns  = len(all_cns)

    if n_cns < 2:
        return {}

    n_rows = len(rows)
    X = lil_matrix((n_rows, n_cns), dtype=np.float32)
    y = np.empty(n_rows, dtype=np.int8)

    for i, r in enumerate(rows):
        l, ri = int(r.left_narrative_id), int(r.right_narrative_id)
        X[i, cn_idx[l]]  =  1.0
        X[i, cn_idx[ri]] = -1.0
        y[i] = int(r.left_chosen)

    # fit_intercept=False: no overall bias; the β are identifiable up to a constant
    clf = LogisticRegression(
        C=1.0, fit_intercept=False, solver='lbfgs',
        max_iter=1000, tol=1e-4
    )
    clf.fit(X.tocsr(), y)

    return {cn: float(clf.coef_[0][cn_idx[cn]]) for cn in all_cns}


# Fit BT model for every (baseclaim, kpi, evaluator) combination
bt_scores = {}
for (bc, kpi, ev), grp in df.groupby(['baseclaim_id', 'kpi_id', 'evaluator_id']):
    bt_scores[(bc, kpi, ev)] = fit_bradley_terry(grp)

# Quick sanity check: how many CNs got a score?
sample_key = (1, 1, 1)
sample = bt_scores[sample_key]
print(f"Example — baseclaim=1, kpi=1, evaluator=1:")
print(f"  {len(sample)} CNs scored")
sorted_sample = sorted(sample.items(), key=lambda x: -x[1])
print(f"  Top-5 CNs:    {sorted_sample[:5]}")
print(f"  Bottom-5 CNs: {sorted_sample[-5:]}")


Example — baseclaim=1, kpi=1, evaluator=1:
  122 CNs scored
  Top-5 CNs:    [(31, 1.1959201182279606), (57, 1.0948741999287788), (34, 1.0288410929489422), (55, 0.9865371676011944), (72, 0.8853661237671265)]
  Bottom-5 CNs: [(3, -0.9010408789732557), (71, -0.905210215056013), (78, -1.0352593491095814), (117, -1.073283128135164), (79, -1.2611014083991596)]


---
## Inter-Rater Agreement — Kendall's W on BT-Implied Rankings

We restrict to CNs observed by **all five evaluators** (~70% per condition, 89–95 CNs) and apply **Kendall's W** to the resulting 5 × n score matrix of BT β scores:

$$W = \frac{12\,S}{m^{2}(n^{3}-n)}$$

where $S$ is the sum of squared deviations of column rank sums, $m = 5$, and $n$ is the number of common CNs.

- **W = 0**: no concordance (evaluators rank CNs independently)
- **W = 1**: perfect agreement (identical rankings)


In [7]:
import numpy as np
from scipy.stats import rankdata, chi2 as chi2_dist


def kendall_w(score_matrix):
    """
    Kendall's W on a (m_raters x n_items) score matrix.
    Returns (W, p_value) using the chi-squared approximation.
    """
    m, n = score_matrix.shape
    ranked = np.array([rankdata(row) for row in score_matrix])
    R = ranked.sum(axis=0)
    S = np.sum((R - R.mean()) ** 2)
    W = 12 * S / (m ** 2 * (n ** 3 - n))
    chi2_stat = m * (n - 1) * W
    p = 1 - chi2_dist.cdf(chi2_stat, df=n - 1)
    return round(W, 4), round(p, 6)


evaluators = sorted(df['evaluator_id'].unique())

bt_w_records = []
for bc in sorted(df['baseclaim_id'].unique()):
    for kpi in sorted(df['kpi_id'].unique()):
        cn_sets    = [set(bt_scores[(bc, kpi, ev)]) for ev in evaluators]
        common_cns = sorted(set.intersection(*cn_sets))

        score_matrix = np.array([
            [bt_scores[(bc, kpi, ev)][cn] for cn in common_cns]
            for ev in evaluators
        ])

        W, p = kendall_w(score_matrix)
        bt_w_records.append({
            'baseclaim_id': bc,
            'kpi_id':       kpi,
            'n_cns':        len(common_cns),
            'kendall_W':    W,
            'p_value':      p,
            'significant':  p < 0.05,
        })

bt_w_df = pd.DataFrame(bt_w_records)

print("=== Kendall's W on BT-Implied Rankings per (base claim, KPI) ===")
print("(W: 0 = no agreement, 1 = perfect; n_cns = CNs seen by all 5 evaluators)\n")
print(bt_w_df.to_string(index=False))

print(f"\n=== Grand mean W: {round(bt_w_df['kendall_W'].mean(), 4)} ===")
print(f"Significant (p<0.05): {bt_w_df['significant'].sum()} / {len(bt_w_df)}")

print("\n=== Mean W per KPI ===")
print(bt_w_df.groupby('kpi_id')['kendall_W'].agg(['mean', 'std']).round(4).to_string())


=== Kendall's W on BT-Implied Rankings per (base claim, KPI) ===
(W: 0 = no agreement, 1 = perfect; n_cns = CNs seen by all 5 evaluators)

 baseclaim_id  kpi_id  n_cns  kendall_W  p_value  significant
            1       1     89     0.4064 0.000000         True
            1       2     89     0.4142 0.000000         True
            1       3     89     0.2569 0.037446         True
            2       1     93     0.3972 0.000000         True
            2       2     93     0.3450 0.000020         True
            2       3     93     0.2040 0.427285        False
            3       1     95     0.3245 0.000129         True
            3       2     95     0.3629 0.000002         True
            3       3     95     0.2506 0.049040         True

=== Grand mean W: 0.3291 ===
Significant (p<0.05): 8 / 9

=== Mean W per KPI ===
          mean     std
kpi_id                
1       0.3760  0.0449
2       0.3740  0.0359
3       0.2372  0.0289


---
## Permutation Baselines

To confirm that observed values significantly exceed chance, we test each metric against a design-appropriate null distribution.

**Rank-biserial r** (per evaluator × base claim × KPI): shuffle `left_chosen` labels within the group, recompute r, repeat N = 1,000 times.

**Kendall's W** (per base claim × KPI): independently permute each evaluator's BT ranking across CNs, recompute W, repeat N = 500 times.

p-values are one-sided: fraction of permutations >= observed.


In [8]:
# Permutation baseline — rank-biserial r
# Null model: shuffle left_chosen labels within each (evaluator, base_claim, KPI)
# group, recompute r.  Copeland scores are re-derived from the shuffled choices,
# so the null reflects the metric's self-consistency floor under random responding.

N_PERM = 1000
_rng   = np.random.default_rng(42)

perm_rbc_rows = []
groups_list   = list(df.groupby(['evaluator_id', 'baseclaim_id', 'kpi_id']))
print(f"Running {N_PERM} permutations x {len(groups_list)} groups (rank-biserial) ...")

for i, ((ev, bc, kpi), grp) in enumerate(groups_list):
    left_ids  = grp['left_narrative_id'].values.astype(int)
    right_ids = grp['right_narrative_id'].values.astype(int)
    choices   = grp['left_chosen'].values.astype(int)
    n         = len(choices)

    obs_rbc = rank_biserial(grp)

    null_rbc = np.empty(N_PERM)
    for j in range(N_PERM):
        perm_c = _rng.permutation(choices)
        wins, losses, outcomes = {}, {}, []
        for k in range(n):
            a, b = left_ids[k], right_ids[k]
            w, l = (a, b) if perm_c[k] == 1 else (b, a)
            wins[w]   = wins.get(w, 0) + 1
            losses[l] = losses.get(l, 0) + 1
            outcomes.append((w, l))
        items = set(wins) | set(losses)
        score = {it: wins.get(it, 0) - losses.get(it, 0) for it in items}
        n_correct = sum(
            1.0 if score[w] > score[l] else 0.5 if score[w] == score[l] else 0.0
            for w, l in outcomes
        )
        null_rbc[j] = ((n_correct / n) - 0.5) * 2

    perm_rbc_rows.append({
        'evaluator_id': ev, 'baseclaim_id': bc, 'kpi_id': kpi,
        'observed':  obs_rbc,
        'p_value':   round(float((null_rbc >= obs_rbc).mean()), 4),
    })

    if (i + 1) % 15 == 0:
        print(f"  {i + 1}/{len(groups_list)} groups complete")

perm_rbc_df = pd.DataFrame(perm_rbc_rows)
sig = (perm_rbc_df['p_value'] < 0.05).sum()
total = len(perm_rbc_df)
print(f"Done. Significant (p < 0.05): {sig} / {total} evaluator x base-claim x KPI conditions")


Running 1000 permutations x 45 groups (rank-biserial) ...
  15/45 groups complete
  30/45 groups complete
  45/45 groups complete
Done. Significant (p < 0.05): 34 / 45 evaluator x base-claim x KPI conditions


In [9]:

# ── Permutation baseline: Kendall's W (BT-implied) ───────────────────────────
# Null model: within each (base_claim, KPI) condition, independently shuffle
# each evaluator's BT ranking across CNs.  This removes concordance while
# preserving each evaluator's marginal score distribution.

N_PERM_W = 500
print(f"Running {N_PERM_W} permutations for Kendall's W (BT-implied) ...")

perm_w_rows = []
for bc in sorted(df['baseclaim_id'].unique()):
    for kpi in sorted(df['kpi_id'].unique()):
        cn_sets    = [set(bt_scores[(bc, kpi, ev)]) for ev in evaluators]
        common_cns = sorted(set.intersection(*cn_sets))

        score_matrix = np.array([
            [bt_scores[(bc, kpi, ev)][cn] for cn in common_cns]
            for ev in evaluators
        ])

        obs_W, _ = kendall_w(score_matrix)
        null_Ws  = np.array([
            kendall_w(np.array([_rng.permutation(row) for row in score_matrix]))[0]
            for _ in range(N_PERM_W)
        ])

        perm_w_rows.append({
            'baseclaim_id': bc,
            'kpi_id':       kpi,
            'observed_W':   obs_W,
            'null_mean_W':  round(float(null_Ws.mean()), 4),
            'null_std_W':   round(float(null_Ws.std()),  4),
            'p_value':      round(float((null_Ws >= obs_W).mean()), 4),
        })

perm_w_df = pd.DataFrame(perm_w_rows)
print("Done.\n")
print(perm_w_df.to_string(index=False))


Running 500 permutations for Kendall's W (BT-implied) ...
Done.

 baseclaim_id  kpi_id  observed_W  null_mean_W  null_std_W  p_value
            1       1      0.4064       0.1990      0.0270    0.000
            1       2      0.4142       0.1978      0.0263    0.000
            1       3      0.2569       0.2012      0.0266    0.024
            2       1      0.3972       0.2005      0.0258    0.000
            2       2      0.3450       0.2006      0.0260    0.000
            2       3      0.2040       0.1979      0.0251    0.398
            3       1      0.3245       0.2004      0.0269    0.000
            3       2      0.3629       0.1990      0.0251    0.000
            3       3      0.2506       0.2009      0.0261    0.038


In [10]:
# ── Permutation baseline summary ─────────────────────────────────────────────
print("=" * 62)
print("PERMUTATION BASELINE SUMMARY")
print("(p-value = fraction of permutations >= observed; one-sided)")
print("=" * 62)

print("\n--- Rank-Biserial r ---")
print(f"  Observed mean:       {perm_rbc_df['observed'].mean():.4f}")
sig_rbc = (perm_rbc_df['p_value'] < 0.05).sum()
print(f"  Significant (p<.05): {sig_rbc} / {len(perm_rbc_df)} conditions")

print("\n--- Kendall's W (BT-implied) ---")
print(f"  Observed mean:       {perm_w_df['observed_W'].mean():.4f}")
sig_w = (perm_w_df['p_value'] < 0.05).sum()
print(f"  Significant (p<.05): {sig_w} / {len(perm_w_df)} conditions")

print("\n--- Rank-biserial r per evaluator (mean across conditions) ---")
print(perm_rbc_df.groupby('evaluator_id')['observed']
      .mean().round(4).rename('rank_biserial').to_string())


PERMUTATION BASELINE SUMMARY
(p-value = fraction of permutations >= observed; one-sided)

--- Rank-Biserial r ---
  Observed mean:       0.7505
  Significant (p<.05): 34 / 45 conditions

--- Kendall's W (BT-implied) ---
  Observed mean:       0.3291
  Significant (p<.05): 8 / 9 conditions

--- Rank-biserial r per evaluator (mean across conditions) ---
evaluator_id
1    0.7739
2    0.7595
3    0.7843
4    0.7830
5    0.6516
